# Sentiment Intelligence Dashboard: Dual-Model NLP Pipeline

This notebook showcases the core AI text intelligence engine powering the full-stack React.js & FastAPI platform. 

We implemented a **dual-model NLP pipeline** combining:
1. **HuggingFace Transformers (`cardiffnlp/twitter-roberta-base-emotion`)**: Deep-learning based emotion detection.
2. **VADER (Lexicon-driven)**: Rule-based sentiment scoring.

Let's see how it works!

In [ ]:
# 1. Install necessary libraries (Uncomment if running locally)
# !pip install transformers vaderSentiment nltk torch pandas

In [ ]:
import nltk
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline
import pandas as pd

# Download NLTK sentence tokenizer
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

## 2. Initialize Models

We load VADER for fast lexicon-based scoring and a fine-tuned RoBERTa model from HuggingFace for deep-learning emotion classification.

In [ ]:
# Initialize VADER
vader_analyzer = SentimentIntensityAnalyzer()

# Initialize HuggingFace Emotion Model
emotion_classifier = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-emotion", top_k=None)

print("Models loaded successfully!")

## 3. Define the Core Analysis Function

This function breaks a document down into sentences and scores each sentence individually using both models. This provides granular insights compared to document-level scoring.

In [ ]:
def analyze_text(text):
    sentences = sent_tokenize(text)
    results = []
    
    for sentence in sentences:
        # VADER Scoring
        vader_scores = vader_analyzer.polarity_scores(sentence)
        
        # HuggingFace Emotion Scoring
        emotions = emotion_classifier(sentence)[0]
        # Format emotions into a dictionary
        emotion_dict = {em['label']: round(em['score'], 4) for em in emotions}
        
        # Determine dominant emotion
        dominant_emotion = max(emotion_dict, key=emotion_dict.get)
        
        results.append({
            "sentence": sentence,
            "vader_compound": vader_scores['compound'],
            "vader_pos": vader_scores['pos'],
            "vader_neg": vader_scores['neg'],
            "vader_neu": vader_scores['neu'],
            "dominant_emotion": dominant_emotion,
            **emotion_dict
        })
        
    return pd.DataFrame(results)

## 4. Test the Pipeline

Let's test it on a sample product review to see how the dual-model approach handles complex sentiment.

In [ ]:
sample_review = """
I absolutely love the new design of this phone! 
However, the battery life is incredibly disappointing and dies by noon. 
Overall, I am feeling very conflicted and frustrated about this purchase.
"""

df_results = analyze_text(sample_review)
df_results.head()

### Interpretation
- **Sentence 1**: High positive VADER score, dominant emotion `joy`.
- **Sentence 2**: Negative VADER score, dominant emotion `sadness` or `anger`.
- **Sentence 3**: Highly negative VADER score, dominant emotion `anger`.

By exposing these granular APIs to the React.js frontend, we are able to build interactive radar charts and dynamic sentence-highlighting views!